# TP4. Matrices et systèmes linéaires

Carnet de travail. L'énoncé complet, avec les corrigés, est sur le site :
https://cours.gclab.fr/outils-fondamentaux/tp4-matrices-gauss/

Exécutez une cellule avec Maj+Entrée. **Pensez à télécharger ce carnet avant de partir** : menu File, puis Download.


## Exercice 1.1 : la transposée

Écrivez `transposee(A)` qui échange lignes et colonnes.

**Résultat attendu :** la transposée de `[[1, 2, 3], [4, 5, 6]]` est
`[[1, 4], [2, 5], [3, 6]]`.


In [ ]:
def transposee(A):
    return [list(colonne) for colonne in zip(*A)]


A = [[1, 2, 3], [4, 5, 6]]
print(transposee(A))

## Exercice 2.1 : produit à la main

Écrivez `produit(A, B)`, et vérifiez sur un exemple que vous savez calculer de tête.

**Résultat attendu :** multiplier une matrice par l'identité la laisse inchangée.


In [ ]:
def produit(A, B):
    if not A or not B:
        return []
    if len(A[0]) != len(B):
        raise ValueError("Dimensions incompatibles")

    resultat = []
    for i in range(len(A)):
        ligne = []
        for j in range(len(B[0])):
            valeur = 0
            for k in range(len(B)):
                valeur += A[i][k] * B[k][j]
            ligne.append(valeur)
        resultat.append(ligne)
    return resultat


A = [[1, 2], [3, 4]]
identite = [[1, 0], [0, 1]]
print(produit(A, identite))

## Exercice 2.2 : mesurer le coût cubique

Mesurez le temps de `produit` sur deux matrices carrées aléatoires de taille 40, 80
puis 120. Que devient le temps quand la taille double ?

**Résultat attendu :** le temps est multiplié par environ 8 quand la taille double.


In [ ]:
import random
import time


for n in (40, 80, 120):
    A = [[random.random() for _ in range(n)] for _ in range(n)]
    B = [[random.random() for _ in range(n)] for _ in range(n)]

    debut = time.perf_counter()
    produit(A, B)
    duree = time.perf_counter() - debut
    print(f"n={n} : {duree:.4f} s")

## Exercice 3.1 : le rapport de temps

Reprenez la mesure de l'exercice 2.2 pour `n = 120`, et comparez au produit numpy des
mêmes matrices.

**Résultat attendu :** numpy est plus rapide de deux à trois ordres de grandeur.


In [ ]:
import numpy as np
import time


n = 120
A = np.random.random((n, n))
B = np.random.random((n, n))

A_list = A.tolist()
B_list = B.tolist()
debut = time.perf_counter()
produit(A_list, B_list)
temps_python = time.perf_counter() - debut

debut = time.perf_counter()
A @ B
temps_numpy = time.perf_counter() - debut

print(f"Python : {temps_python:.4f} s")
print(f"NumPy : {temps_numpy:.6f} s")
print(f"NumPy est environ {temps_python / temps_numpy:.0f} fois plus rapide.")

## Exercice 4.1 : élimination et remontée

Écrivez `resoudre(A, b)` qui résout le système par le pivot de Gauss avec pivot
partiel, et testez-la sur le système de l'étape 1.

**Résultat attendu :** la solution est $(3,\, -1,\, 0{,}5)$, et $A x$
redonne bien $b$.


In [ ]:
def resoudre(A, b):
    n = len(A)
    M = [list(A[i]) + [b[i]] for i in range(n)]

    for k in range(n):
        ligne_pivot = max(range(k, n), key=lambda i: abs(M[i][k]))
        if M[ligne_pivot][k] == 0:
            raise ValueError("Système singulier")
        M[k], M[ligne_pivot] = M[ligne_pivot], M[k]

        for i in range(k + 1, n):
            facteur = M[i][k] / M[k][k]
            for j in range(k, n + 1):
                M[i][j] -= facteur * M[k][j]

    solution = [0] * n
    for i in range(n - 1, -1, -1):
        reste = M[i][n]
        for j in range(i + 1, n):
            reste -= M[i][j] * solution[j]
        solution[i] = reste / M[i][i]

    return solution


A = [[1, 2, 2], [1, 3, -2], [3, 5, 8]]
b = [2, -1, 8]
solution = resoudre(A, b)
print(solution)
print([sum(A[i][j] * solution[j] for j in range(3)) for i in range(3)])

## Exercice 4.2 : confronter à numpy

Comparez votre solution à celle de `numpy.linalg.solve` sur un système aléatoire de
taille 50.

**Résultat attendu :** les deux solutions coïncident à $10^{-9}$ près.


In [ ]:
import numpy as np


n = 50
A = np.random.random((n, n)) + n * np.eye(n)
x_vrai = np.random.random(n)
b = A @ x_vrai

solution_gauss = np.array(resoudre(A.tolist(), b.tolist()))
solution_numpy = np.linalg.solve(A, b)

print("Solutions égales à 1e-9 près :", np.allclose(solution_gauss, solution_numpy, atol=1e-9))

## Exercice 5.1 : lire le résultat

Exécutez le bloc ci-dessus et commentez. Quelle méthode donne la solution la plus
proche de la vraie ? Que vaut le conditionnement, et qu'est-ce que cela signifie ?


In [ ]:
import numpy as np


n = 12
H = np.array([[1.0 / (i + j + 1) for j in range(n)] for i in range(n)])
x_vrai = np.ones(n)
b = H @ x_vrai

solution = np.linalg.solve(H, b)
solution_inverse = np.linalg.inv(H) @ b
erreur_solve = np.max(np.abs(solution - x_vrai))
erreur_inverse = np.max(np.abs(solution_inverse - x_vrai))
conditionnement = np.linalg.cond(H)

print(f"Erreur de solve : {erreur_solve:.3e}")
print(f"Erreur de inv(A) @ b : {erreur_inverse:.3e}")
print(f"Conditionnement : {conditionnement:.3e}")
print("solve est généralement plus précis ; le grand conditionnement signifie que de petites erreurs sur les données peuvent produire de grandes erreurs sur la solution.")

## Exercice 6.1 : composer deux transformations

Vérifiez numériquement que faire tourner de 30 degrés puis de 60 degrés revient à
faire tourner de 90 degrés, autrement dit que
$R(60°) \cdot R(30°) = R(90°)$.


In [ ]:
import numpy as np


def rotation(angle):
    c = np.cos(angle)
    s = np.sin(angle)
    return np.array([[c, -s], [s, c]])


resultat = rotation(np.pi / 3) @ rotation(np.pi / 6)
attendu = rotation(np.pi / 2)
print(np.allclose(resultat, attendu))

## Exercice 7.1 : déterminant par Gauss

Écrivez `determinant(A)` en reprenant l'élimination de `resoudre`, sans second membre :
multipliez les pivots successifs, et changez le signe à chaque échange de lignes.
Comparez à `np.linalg.det`.

**Résultat attendu :** le déterminant de la matrice de l'étape 1 vaut `-2`, et celui
de `[[1, 2], [2, 4]]` vaut `0`.


In [ ]:
import numpy as np


def determinant(A):
    n = len(A)
    M = [list(ligne) for ligne in A]
    resultat = 1

    for k in range(n):
        ligne_pivot = max(range(k, n), key=lambda i: abs(M[i][k]))
        if M[ligne_pivot][k] == 0:
            return 0
        if ligne_pivot != k:
            M[k], M[ligne_pivot] = M[ligne_pivot], M[k]
            resultat *= -1

        pivot = M[k][k]
        resultat *= pivot
        for i in range(k + 1, n):
            facteur = M[i][k] / pivot
            for j in range(k + 1, n):
                M[i][j] -= facteur * M[k][j]

    return resultat


A = [[1, 2, 2], [1, 3, -2], [3, 5, 8]]
print(determinant(A))
print(determinant([[1, 2], [2, 4]]))
print(np.linalg.det(A))

## Exercice 8.1 : Hilbert vaincue

Construisez la matrice de Hilbert de taille 12 avec des `Fraction(1, i + j + 1)`, le
second membre `b` comme somme des lignes, pour que la solution soit le vecteur de 1,
puis appelez **votre** fonction `resoudre` de l'étape 4 dessus. Le premier essai échoue
avec « système singulier ». Comprenez pourquoi avant de lire le corrigé, puis corrigez.

**Résultat attendu :** exactement `[1, 1, ..., 1]`, douze fois `Fraction(1, 1)`, là
où `solve` se trompait de 0,7.


In [ ]:
from fractions import Fraction


n = 12
H = [
    [Fraction(1, i + j + 1) for j in range(n)]
    for i in range(n)
]
b = [sum(ligne) for ligne in H]
solution = resoudre(H, b)

print(solution)
assert solution == [Fraction(1, 1)] * n

## Exercice 9.1 : l'ordre compte

Écrivez `translation(tx, ty)` et `rotation_h(angle)` qui renvoient des matrices
$3 \times 3$. Appliquez au point $(1, 0)$ la translation de $(3, 1)$ suivie
de la rotation de 90°, puis la rotation suivie de la translation.

**Résultat attendu :** $(-1, 4)$ dans le premier cas, $(3, 2)$ dans le second.
Les deux résultats diffèrent.


In [ ]:
import numpy as np


def translation(tx, ty):
    return np.array([[1, 0, tx], [0, 1, ty], [0, 0, 1]])


def rotation_h(angle):
    c = np.cos(angle)
    s = np.sin(angle)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])


point = np.array([1, 0, 1])
T = translation(3, 1)
R = rotation_h(np.pi / 2)

print((R @ T @ point)[:2])
print((T @ R @ point)[:2])